In [ ]:
        {
            "cell_type": "code",
            "id": "#VSC-1e6ce502",
            {
                "cells": [
                    {
                        "cell_type": "code",
                        "id": "#VSC-8d58e66b",
                        "metadata": {
                            "language": "python"
                        },
                        "source": [
                            "import numpy as np",
                            "import sage.all as sage",
                            "from sage.all import var, sin, cos, pi, ln, diff",
                            "from ode_viewer_nD import ODESystemND, ODEViewerND, generate_ic_grid"
                        ],
                        "execution_count": null,
                        "outputs": []
                    },
                    {
                        "cell_type": "code",
                        "id": "#VSC-c933f199",
                        "metadata": {
                            "language": "python"
                        },
                        "source": [
                            "# CONFIGURATION VARIABLES",
                            "",
                            "# --- Numerical Settings ---",
                            "t_start = -5.0  # Start time",
                            "t_end = 50.0  # End time",
                            "num_points = 1000  # Number of points for trajectory (higher = smoother)",
                            "step_size = 0.01  # Integration step size",
                            "",
                            "# --- Initial Conditions ---",
                            "ic_center = [0.0, 0.0, 0.0]  # Center of initial condition grid",
                            "ic_spread = 2.0  # Spread around center (by unit)",
                            "num_ics_x = 2  # Number of ICs along x direction",
                            "num_ics_y = 2  # Number of ICs along y direction",
                            "num_ics_z = 2  # Number of ICs along z direction",
                            "",
                            "# N-D initial-condition parameters (None falls back to legacy vars)",
                            "ic_centers = [0.0, 0.0, 0.0, 0.0]  # e.g. [0.0, 0.0, 0.0] for 3 vars or None to auto",
                            "ic_spreads = None  # e.g. [1.0, 1.0, 1.0] or None to use `ic_spread`",
                            "num_ics_per_axis = [2, 2, 2, 2]  # e.g. [3,3,3] or None to use `num_ics_x/y/z`",
                            "",
                            "# --- Projection Settings ---",
                            "# Which 3 variables to use as (x_axis, y_axis, z_axis) in the 3D plot",
                            "# 0 = time, 1 = x, 2 = dx/dt, 3 = d²x/dt², 4 = d³x/dt³, ...",
                            "# projection_axes = [1, 2, 3]",
                            "projection_axes = [1, 2, 3]",
                            "",
                            "# --- Display Settings ---",
                            "color_by = 0  # Which variable to color by (0 = t, 1 = x, 2 = dx/dt, ...) or None",
                            "figsize = (10, 8)  # Figure size (width, height)",
                            "title = \"Phase Space\"  # Plot title",
                            "max_boundary = 10.0  # Maximum axis limit for plot boundaries (None = auto)",
                            "compute_boundary = 2 * max_boundary  # Stop computation when values exceed this (None = no limit)",
                            "",
                            "# --- Output Settings ---",
                            "save_plot = True  # Save the plot to a file",
                            "save_path = \"./Plots/3D/3D_NSYS_ODE.png\"",
                            "",
                            "# --- Live Viewer Settings ---",
                            "use_threejs = True",
                            "threejs_filename = \"./Plots/3D/3D_NSYS_ODE.html\"",
                            "",
                            "",
                            "def get_color(val):",
                            "    r = max(0, min(1, 2 * val))",
                            "    g = max(0, min(1, 2 * (1 - val)))",
                            "    b = max(0, min(1, 2 * (1 - abs(val - 0.5))))",
                            "    return (r, g, b)",
                            ""
                        ],
                        "execution_count": null,
                        "outputs": []
                    },
                    {
                        "cell_type": "code",
                        "id": "#VSC-38b06b36",
                        "metadata": {
                            "language": "python"
                        },
                        "source": [
                            "# SYSTEM DEFINITION",
                            "",
                            "# --- System Definition ---",
                            "# For first-order systems (like Lorenz): use first_order_system = True",
                            "first_order_system = False  # Set True for systems like Lorenz (dx/dt = ...)",
                            "# Otherwise, for higher-order ODEs: define dⁿx/dtⁿ",
                            "ode_order = 4  # Order of the ODE (2, 3, 4, ...)",
                            "",
                            "if first_order_system:",
                            "    var_names = [\"x\", \"y\", \"z\"]  # [x₁, x₂, x₃, ..., xₙ]",
                            "    n_vars = len(var_names)  # Number of variables",
                            "",
                            "    # First-order systems",
                            "    # f: t, x₁, x₂, x₃, ..., xₙ -> R × R × ... × R",
                            "    def system_func(t, state):",
                            "        # state = [x₁, x₂, x₃, ..., xₙ]",
                            "        x, y, z = state",
                            "",
                            "        # sigma, rho, beta = 10, 28, 8 / 3",
                            "        # dx = sigma * (y - x)",
                            "        # dy = x * (rho - z) - y",
                            "        # dz = x * y - beta * z",
                            "",
                            "        dx = sin(y)",
                            "        dy = sin(z)",
                            "        dz = sin(x)",
                            "",
                            "        return [dx, dy, dz]  # returns: [dx1, dx2, dx3, ..., dxn]",
                            "",
                            "else:",
                            "    n_vars = ode_order",
                            "    var_names = [\"x\"] + [f\"x^{(i)}\" for i in range(1, ode_order)]",
                            "",
                            "    # Higher order systems",
                            "    # dⁿx/dtⁿ: t, x, x/dt, d²x/dt², ... -> R",
                            "    def highest_derivative(t, state):",
                            "        # state = [x, dx/dt, d²x/dt², ..., dⁿx/dtⁿ)",
                            "        x, dx, ddx, dddx = state",
                            "",
                            "        return x * dddx + t ** 2  # returns: dⁿx/dtⁿ"
                        ],
                        "execution_count": null,
                        "outputs": []
                    },
                    {
                        "cell_type": "code",
                        "id": "#VSC-1e6ce502",
                        "metadata": {
                            "language": "python"
                        },
                        "source": [
                            "# RUNNING AND PLOTTING",
                            "",
                            "if first_order_system:",
                            "    system = ODESystemND(system_func, n_vars, var_names)",
                            "else:",
                            "    system = ODESystemND.from_higher_order(highest_derivative, ode_order, var_names)",
                            "",
                            "# Build initial-condition grid automatically for n_vars (use the N-D helper).",
                            "if ic_centers is None:",
                            "    if 'ic_center' in globals() and len(ic_center) == n_vars:",
                            "        ic_centers = ic_center",
                            "    else:",
                            "        ic_centers = [0.0] * n_vars",
                            "if ic_spreads is None:",
                            "    ic_spreads = [ic_spread] * n_vars",
                            "if num_ics_per_axis is None:",
                            "    try:",
                            "        num_ics_per_axis = [num_ics_x, num_ics_y, num_ics_z][:n_vars]",
                            "    except NameError:",
                            "        num_ics_per_axis = [2] * n_vars",
                            "",
                            "ic_grid = generate_ic_grid(ic_centers, ic_spreads, num_ics_per_axis)",
                            "t_eval = np.linspace(t_start, t_end, num_points)",
                            "viewer = ODEViewerND()",
                            "viewer.solve_ics_grid(",
                            "    system,",
                            "    (t_start, t_end),",
                            "    ic_grid,",
                            "    len(ic_grid),",
                            "    t_eval=t_eval,",
                            ")",
                            "",
                            "viewer.plot_all_3d(",
                            "    projection_axes=projection_axes,",
                            "    color_variable=color_by,",
                            "    figsize=figsize,",
                            "    title=title,",
                            "    save_path=save_path if save_plot else None,",
                            "    max_boundary=max_boundary,",
                            "    compute_boundary=compute_boundary,",
                            ")",
                            "",
                            "print(f\"Plotted {len(viewer.solutions)} trajectories.\")",
                            "print(f\"Projection: {projection_axes}\")"
                        ],
                        "execution_count": null,
                        "outputs": []
                    },
                    {
                        "cell_type": "code",
                        "id": "#VSC-1ccefdb7",
                        "metadata": {
                            "language": "python"
                        },
                        "source": [
                            "# THREE.JS INTERACTIVE VIEWER",
                            "",
                            "if use_threejs:",
                            "    traj_data = viewer.get_threejs_trajectories(",
                            "        projection_axes=projection_axes,",
                            "        color_variable=color_by,",
                            "        compute_boundary=compute_boundary,",
                            "    )",
                            "",
                            "    plot_obj = sage.Graphics()",
                            "",
                            "    for idx, traj in enumerate(traj_data):",
                            "        points = traj[\"points\"]",
                            "",
                            "        if color_by is not None and traj[\"normalized_colors\"] is not None:",
                            "            colors = traj[\"normalized_colors\"]",
                            "            for i in range(0, len(points) - 1):",
                            "                color_val = colors[i]",
                            "                segment = sage.line3d(",
                            "                    [points[i], points[i + 1]],",
                            "                    color=sage.hue(color_val),",
                            "                    thickness=2,",
                            "                    opacity=0.8,",
                            "                )",
                            "                plot_obj += segment",
                            "        else:",
                            "            traj_color = traj.get(\"trajectory_color\", 0)",
                            "            line = sage.line3d(",
                            "                points,",
                            "                color=sage.hue(traj_color),",
                            "                thickness=2,",
                            "                opacity=0.8,",
                            "            )",
                            "            plot_obj += line",
                            "        print(f\"Completed trajectory {idx + 1} of {len(traj_data)}\")",
                            "        ",
                            "    if color_by is not None:",
                            "        print(",
                            "            f\"Color by: {color_by} ({['time', 'x', 'dx/dt', 'd²x/dt²', 'd³x/dt³'][color_by] if color_by < 5 else 'custom'})\"",
                            "        )",
                            "",
                            "    plot_obj.save(threejs_filename, viewer=\"threejs\", online=True)",
                            "    print(f\"Three.js plot saved to: {threejs_filename}\")",
                            "    sage.show(plot_obj)"
                        ],
                        "execution_count": null,
                        "outputs": []
                    }
                ],
                "metadata": {},
                "nbformat": 4,
                "nbformat_minor": 5
            }